In [1]:
import torch
import numpy as np
import time

def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using NVIDIA CUDA GPU


In [33]:
from helper import load_model_weights
from pathlib import Path
from Qwen3 import Qwen3Model, QWEN_CONFIG_06_B, QWEN_CONFIG_4_B_INSTRUCT

model_name = "Qwen/Qwen3-0.6B"
model_config = QWEN_CONFIG_06_B
tokenizer_path = Path("qwen3") / "qwen3-0.6B-base.json"
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"


In [34]:
tokenizer, model = load_model_weights(None, None, model_config, model_name, 'qwen3/')

Loading tokenizer from None...
Tokenizer Path does not exist, loading using model_name Qwen/Qwen3-0.6B
Saving tokenizer to qwen3/...
Tokenizer saved to qwen3/Qwen/Qwen3-0.6B.json
Using {'vocab_size': 151936, 'context_length': 40960, 'emb_dim': 1024, 'n_heads': 16, 'n_layers': 28, 'hidden_dim': 3072, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 1000000.0, 'dtype': torch.bfloat16} load model...
Successfully loaded model from config {'vocab_size': 151936, 'context_length': 40960, 'emb_dim': 1024, 'n_heads': 16, 'n_layers': 28, 'hidden_dim': 3072, 'head_dim': 128, 'qk_norm': True, 'n_kv_groups': 8, 'rope_base': 1000000.0, 'dtype': torch.bfloat16}.
Loading weights from None...
Model does not exist, loading using model_name Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Successfully loaded model from Qwen/Qwen3-0.6B
Saving model to qwen3/...
Model successfully saved to qwen3/Qwen/Qwen3-0.6B.pth


In [35]:
model = model.to(device)

In [36]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

In [37]:
def prep_data_train(tokenizer, data):
    tokenizer.padding_side = 'right'
    n = len(data['Question'])
    l = []
    for i in range(n):
        conv = []
        conv.append({
            'role': 'user', 'content': ds['train'][i]['Question'] + ' /think',
        })
        conv.append({
            'role': 'assistant', 'content': '<think>\n' + ds['train'][i]['Complex_CoT'] + '\n</think>\n\n' +  ds['train'][i]['Response']
        })
        l.append(conv)
    return tokenizer.apply_chat_template(l, return_tensors='pt', padding=True, truncation=True)
    

In [38]:
def prep_data_inference(tokenizer, data, enable_thinking=False):
    tokenizer.padding_side='left'
    n = len(data['Question'])
    l = []
    
    for i in range(n):
        conv = []
        if enable_thinking:
            conv.append({
                'role': 'user', 'content': ds['train'][i]['Question'] + ' /think',
            })
        else:   
            conv.append({
                'role': 'user', 'content': ds['train'][i]['Question'],
            })
        l.append(conv)
    return tokenizer.apply_chat_template(l, return_tensors='pt', padding=True, truncation=True)

In [39]:
q_tokens = prep_data_inference(tokenizer, ds['train'][:3]).to(device)

In [40]:
tokenizer.decode(q_tokens['input_ids'])

['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|im_start|>user\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain 

In [30]:
from Qwen3 import KVCache

@torch.inference_mode()
def generate_text_greedy_cache(
    model,
    token_ids,
    max_new_tokens=100,
    eos_token_id=None
):
    input_length = token_ids.shape[1]
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    # cache = KVCache(n_layers=model.config.num_hidden_layers)
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = out.argmax(dim=-1, keepdim=True)
        if (eos_token_id is not None and (next_token == eos_token_id).all()):
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)
        out = model(next_token, cache=cache)[:, -1]
    return token_ids[:, input_length:]    

In [59]:
input_tokens['input_ids']

tensor([[151644,    872,    198,  ..., 151643, 151643, 151643],
        [151644,    872,    198,  ...,     13, 151645,    198]],
       device='cuda:0')

In [62]:
model.generate?

Signature:
model.generate(
    inputs: torch.Tensor | None = None,
    generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None,
    logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None,
    stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None,
    prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None,
    synced_gpus: bool | None = None,
    assistant_model: ForwardRef('PreTrainedModel') | None = None,
    streamer: ForwardRef('BaseStreamer') | None = None,
    negative_prompt_ids: torch.Tensor | None = None,
    negative_prompt_attention_mask: torch.Tensor | None = None,
    custom_generate: str | collections.abc.Callable | None = None,
    **kwargs,
) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generation.utils.GenerateEncoderDecoderOutput | transformers.generation.utils.GenerateBeamDecoderOnlyO

In [77]:
input_tokens['input_ids'][1]

tensor([151644,    872,    198,     32,    220,     18,     18,   4666,   6284,
          5220,    374,   7117,    311,    279,  12851,   9292,    220,     16,
            20,   4420,   1283,   1660,  50180,    304,    279,  15138,    448,
           264,  21966,  12521,     13,  16246,   1059,  16198,  11929,    315,
         27235,    220,     16,     16,     15,  44173,     11,  32415,    804,
           220,     17,     17,  44173,     11,    323,   6543,   7262,    220,
            24,     15,     14,     21,     20,   9465,    472,     70,     11,
          3156,    448,    279,   9362,    315,    264,    220,     20,   1786,
            76,   5538,  26964,  26555,    518,    279,   8416,   3886,    315,
           279,    220,     23,    339,  20131,    304,    279,   2115,   5099,
           706,  34505,   1555,     11,    892,  74793,    938,   5944,    304,
          1059,  15138,    374,   1429,   4363,    311,    387,  15532,     30,
           608,  26865, 151645,    198, 

In [67]:
tokenizer.decode(input_tokens['input_ids'])

["<|im_start|>user\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings? /think<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot 

In [69]:
model.generate_batch?

Signature:
model.generate_batch(
    inputs: list[list[int]],
    generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None,
    num_q_padding_intervals: int = 0,
    num_kv_padding_intervals: int = 0,
    allow_block_sharing: bool = True,
    record_timestamps: bool = False,
    progress_bar: bool = True,
    **kwargs,
) -> dict[str, transformers.generation.continuous_batching.requests.GenerationOutput]
Docstring:
Generate sequences for a batch of prompts using continuous batching.

Args:
    inputs: List of input token sequences (prompts)
    generation_config: Optional generation configuration
    num_q_padding_intervals: Number of intervals used to pad the query dimension
    num_kv_padding_intervals: Number of intervals used to pad the keys/values dimension
    allow_block_sharing: A flag to allow block sharing if the model has some full attention layers
    record_timestamps: If set to true, the requests will have a timestamp for each token gen

In [79]:
model.generate?

Signature:
model.generate(
    inputs: torch.Tensor | None = None,
    generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None,
    logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None,
    stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None,
    prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None,
    synced_gpus: bool | None = None,
    assistant_model: ForwardRef('PreTrainedModel') | None = None,
    streamer: ForwardRef('BaseStreamer') | None = None,
    negative_prompt_ids: torch.Tensor | None = None,
    negative_prompt_attention_mask: torch.Tensor | None = None,
    custom_generate: str | collections.abc.Callable | None = None,
    **kwargs,
) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generation.utils.GenerateEncoderDecoderOutput | transformers.generation.utils.GenerateBeamDecoderOnlyO

In [96]:
res = model.generate(inputs=input_tokens['input_ids'][1:,:112], max_new_tokens=100)

In [97]:
res.shape

torch.Size([1, 212])

In [95]:
tokenizer.decode(input_tokens['input_ids'][1:,:112])

['<|im_start|>user\nA 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured? /think<|im_end|>\n']

In [98]:
tokenizer.decode(res)

["<|im_start|>user\nA 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured? /think<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's figure out what's going on here. A woman comes in with a stab wound from a screwdriver. It's in her chest, upper border of the 8th rib, left side, kind of around the midaxillary line. First thought, that's pretty close to where the lung sits, right?\n\nLet's talk about location first. This spot is along the left side of her body. Above the 8th rib, like that,"]

In [100]:
tokenizer.decode(input_tokens['input_ids'][1:])

["<|im_start|>user\nA 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured? /think<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's figure out what's going on here. A woman comes in with a stab wound from a screwdriver. It's in her chest, upper border of the 8th rib, left side, kind of around the midaxillary line. First thought, that's pretty close to where the lung sits, right?\n\nLet's talk about location first. This spot is along the left side of her body. Above the 8th rib, like that, is where a lot of important stuff lives, like the bottom part of the left lung, possibly the diaphragm too, especially considering how deep the screwdriver

In [66]:
tokenizer.padding_side='left'
tokenizer.decode(model.generate(**input_tokens))

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


["<|im_start|>user\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings? /think<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot 

In [40]:
for i in range(3):
    print("------------------------------------------------------------------")
    print('Question:', ds['train'][i]['Question'])
    print('Correct Answer:', ds['train'][i]['Response'])
    print('LLM Answer:', tokenizer.decode(res[i]))
    

------------------------------------------------------------------
Question: Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?
Correct Answer: The specific cardiac abnormality most likely to be found in this scenario is a patent foramen ovale (PFO). This condition could allow a blood clot from the venous system, such as one from a deep vein thrombosis in the leg, to bypass the lungs and pass directly into the arterial circulation. This can occur when the clot moves from the right atrium to the left atrium through the PFO. Once in the arterial system, the clot can travel to the brain, potentially causing an embolic stroke, which would explain the sudden weakness in the left arm and leg. The connection between the recent travel, which increases the risk of deep vein thrombo

In [43]:
ds_split = ds['train'].train_test_split(train_size=0.8)

In [47]:
from torch import nn
batch_size = 2
n_epochs = 2
n_train = 13760
n_batches = n_train // batch_size
loss = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
losses = []

for i in range(n_epochs):
    print(f"Start {i} epoch----------------------------------------------------")
    ds_split['train'].shuffle()
    for j in range(n_batches):
        start_idx = j * batch_size
        end_idx = (j+1) * batch_size
        input_tokens = prep_data_train(tokenizer, ds_split['train'][start_idx:end_idx]).to(device)
        mask = torch.cumsum(input_tokens['input_ids'] == 151644, dim=-1)>=2
        
        logits = model(**input_tokens).logits[:,:-1]
        targets = input_tokens['input_ids'][:,1:]
        
        opt.zero_grad()
        cur_loss = loss(logits[mask[:, 1:]], targets[mask[:, 1:]])
        losses.append(cur_loss.item())
        cur_loss.backward()
        opt.step()
        if j%10 == 0:
            print(f"Loss = {np.array(losses).mean()}")
                
        

Start 0 epoch----------------------------------------------------
Loss = 2.015625
Loss = 0.38574288108132104
Loss = 0.20233308701288133
Loss = 0.13713762837071572
Loss = 0.10372885262093894
Loss = 0.08341748106713388
Loss = 0.06976382458796267
Loss = 0.05995517381480042
Loss = 0.052567770451675225
Loss = 0.04680335390698779
Loss = 0.04217990082089264
Loss = 0.038389150086823885
Loss = 0.035224638694574026
Loss = 0.032543007654088144
Loss = 0.030241509701343292
Loss = 0.028244650127082473
Loss = 0.026495696594996482
Loss = 0.024951137297334728
Loss = 0.023577118446813764
Loss = 0.02234687605453411
Loss = 0.02123895094762394
Loss = 0.020235931703829653
Loss = 0.01932362086093264
Loss = 0.018490219529056964
Loss = 0.01772591879753651
Loss = 0.01702245013172408
Loss = 0.016372841436744192
Loss = 0.015771121556468554
Loss = 0.01521218798762963
Loss = 0.014691621577207166
Loss = 0.014205601524276986
Loss = 0.013750810715163252
Loss = 0.013324317159682419
Loss = 0.012923565515938842
Loss = 0.

KeyboardInterrupt: 

In [51]:
losses[-1]

3.504753112792969e-05

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 3.2031,  3.7344,  4.6875,  ...,  1.8047,  1.8047,  1.8047],
         [-2.0469, -1.7031,  3.2031,  ..., -2.0469, -2.0469, -2.0469],
         [ 1.9453, -1.0547,  1.9688,  ...,  2.1406,  2.1406,  2.1406],
         ...,
         [ 1.4297, -0.4727, -2.3438,  ..., -1.2344, -1.2344, -1.2344],
         [ 0.5820, -1.7812, -2.9375,  ..., -1.3047, -1.3047, -1.3047],
         [ 1.5938, -1.7422, -2.8125,  ..., -1.2031, -1.2031, -1.2031]],

        [[ 3.2031,  3.7344,  4.6875,  ...,  1.8047,  1.8047,  1.8047],
         [-2.0469, -1.7031,  3.2031,  ..., -2.0469, -2.0469, -2.0469],
         [ 1.9453, -1.0547,  1.9688,  ...,  2.1406,  2.1406,  2.1406],
         ...,
         [ 9.1250, -3.4375,  6.4062,  ..., -0.9609, -0.9609, -0.9609],
         [ 9.9375,  6.4375,  6.2500,  ..., -4.1562, -4.1562, -4.1562],
         [ 7.9375,  3.5781,  8.8750,  ..., -0.0315, -0.0315, -0.0315]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>

In [68]:
tokenizer.decode(input_tokens['input_ids'][0][mask[0]])

"<|im_start|>assistant\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the left side of the heart, it could shoot off to the brain and cause that sudden weakness by blocking blood flow there.\n\nHmm, but how would the clot get from the right side of the heart to the left without going through the lungs and getting filtered out?\n\nHere's wher

In [62]:
q_tokens['input_ids'] = torch.tensor(q_tokens['input_ids']).to(device)
q_tokens['attention_mask'] = torch.tensor(q_tokens['attention_mask']).to(device)

In [63]:
q_tokens['attention_mask']

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1,

In [64]:
cot_tokens['input_ids'] = torch.tensor(cot_tokens['input_ids']).to(device)
cot_tokens['attention_mask'] = torch.tensor(cot_tokens['attention_mask']).to(device)

In [67]:
cot_tokens['attention_mask']

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')

In [76]:
ds['train'][:3]['Question']

['Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
 'A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?',
 'A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?']

In [78]:
tokenizer.decode(q_tokens['input_ids'])

['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',


In [93]:
tokenizer.decode(q_tokens['input_ids'][:, :-2])

['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these',
 'A 33-yea

In [95]:
[tokenizer.decode(x) for x in (model(**q_tokens).logits[:,-3,:]).argmax(dim=-1)]

[' symptoms', ' injured', 'actions']

In [10]:
from torch import nn

class Qwen3SFT(nn.Module):
    def __init__(self):

    def forward(self, x):
        
    

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 19704
    })
})